In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_algorithms.gradients import ReverseEstimatorGradient
from IPython.display import display, Markdown
import warnings

warnings.filterwarnings("ignore", message=".*Casting complex values to real.*")

In [2]:
def ansatz_Odra(n_qubits, depth):
    # --- SPECJALNY PRZYPADEK DLA DEPTH = 1 ---
    if depth == 1:
        qc = QuantumCircuit(n_qubits)
        depth = 1
        params_per_iter = 2 * n_qubits
        theta = ParameterVector('θ', params_per_iter)

        qc = QuantumCircuit(n_qubits)
            # -------- Layer 1: Ry + Ring Cz (with Rz) --------

            # 1. Independent Ry rotations
        for i in range(n_qubits):
            qc.ry(theta[i], i)

            # 2. Ring entanglement (0-1, 1-2, ..., n-1-0)
            # Each pair: Rz on target + Cz
        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits

            param_idx = n_qubits + i
            qc.rz(theta[param_idx], target)
            qc.cz(control, target)

        return qc

    params_per_iter = 4 * n_qubits
    theta = ParameterVector('θ', params_per_iter * (depth // 2))

    qc = QuantumCircuit(n_qubits)

    for j in range(depth // 2):
        offset = j * params_per_iter

        # -------- Layer 1: Ry + Ring Cz (with Rz) --------

        # 1. Independent Ry rotations
        for i in range(n_qubits):
            qc.ry(theta[offset + i], i)

        # 2. Reverse ring entanglement (or shifted)
        # Each pair: Ry on target + Cz
        for i in range(n_qubits):
            control = i
            target = (i - 1) % n_qubits

            param_idx = n_qubits + i
            qc.rz(theta[param_idx], target)
            qc.cz(control, target)

        # -------- Layer 2: Rx + Ring Cz (with Ry) --------

        offset_l2 = offset + 2 * n_qubits

        # 1. Independent Rx rotations
        for i in range(n_qubits):
            qc.rx(theta[offset_l2 + i], i)

        # 2. Reverse ring entanglement (or shifted)
        # Each pair: Ry on target + Cz
        i_15 = 15 - offset_l2 - n_qubits
        # 1. opening of the ring with a single pair of gates
        if 0 <= i_15 < n_qubits:
            control_15 = i_15
            target_15 = (i_15 - 1) % n_qubits
            
            qc.cz(control_15, target_15)
            qc.ry(theta[15], target_15)

        # 2. reverced ring
        for i in reversed(range(n_qubits)):
            if i == i_15:
                continue 
                
            control = i
            target = (i - 1) % n_qubits
            param_idx = offset_l2 + n_qubits + i
            
            qc.cz(control, target)
            qc.ry(theta[param_idx], target)

    return qc

In [3]:
class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit, num_qubits):
        super().__init__()
        self.feature_map = self._create_angle_encoding(num_qubits)
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)
        
        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
        
        estimator = StatevectorEstimator()
        gradient = ReverseEstimatorGradient(estimator)
        
        self.qnn = EstimatorQNN(
            circuit=self.qc, observables=observable, input_params=input_params,
            weight_params=weight_params, estimator=estimator, gradient=gradient
        )
        self.quantum_layer = TorchConnector(self.qnn)
    
    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.quantum_layer(x)

class HybridModelNoise(nn.Module):
    def __init__(self, ansatz_circuit, num_qubits, epsilon=0.005):
        super().__init__()
        self.feature_map = self._create_angle_encoding(num_qubits)
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)
        
        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
        
        estimator = StatevectorEstimator()
        gradient = ReverseEstimatorGradient(estimator)
        
        self.qnn = EstimatorQNN(
            circuit=self.qc, observables=observable, input_params=input_params,
            weight_params=weight_params, estimator=estimator, gradient=gradient
        )
        self.quantum_layer = TorchConnector(self.qnn)

        N_g = 10  
        L = 2     
        self.p_error = 1 - (1 - epsilon) ** (N_g * L)
        self.sigma_noise = 0.2 * self.p_error

    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f_noiseless = self.quantum_layer(x)
        if self.training:
            attenuation = 1.0 - self.p_error
            noise = torch.randn_like(f_noiseless) * self.sigma_noise
            return (attenuation * f_noiseless) + noise
        else:
            return f_noiseless

In [4]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display, Markdown

NUM_QUBITS = 5
K_FOLDS = 5

# --- Automatyczne wykrywanie katalogów depth_n ---
DEPTHS = []
for item in os.listdir('.'):
    if os.path.isdir(item) and item.startswith('depth_'):
        try:
            # Wyciągamy liczbę z nazwy folderu (np. "depth_2" -> 2)
            depth_val = int(item.split('_')[1])
            DEPTHS.append(depth_val)
        except ValueError:
            pass # Ignorujemy foldery, które nie mają poprawnej liczby

DEPTHS.sort() # Sortujemy rosnąco, żeby tabela ładnie wyglądała (np. 2, 4, 6)

print(f"Wykryte głębokości: {DEPTHS}")

if not DEPTHS:
    print("Nie znaleziono żadnych folderów pasujących do wzorca 'depth_n'.")
else:
    # Nagłówek wspólnej tabeli
    markdown_table = """| Głębokość (Depth) | Metryka | Model Idealny (Noiseless) | Model Zaszumiony (Noisy) | Różnica (Noisy vs Idealny) |
| :--- | :--- | :--- | :--- | :--- |
"""

    # Zmienna na podsumowanie acc+f1 (format TSV idealny do wklejenia do Excela)
    summary_lines = "\n### Podsumowanie (Mean Accuracy + Mean F1 Score)\n"
    summary_lines += "Gotowe do przekopiowania do Excela:\n"
    summary_lines += "```text\n"
    summary_lines += "Depth\tIdeal_Sum\tNoisy_Sum\tDiff_Sum\n"

    for depth in DEPTHS:
        ideal_accs, ideal_f1s = [], []
        noisy_accs, noisy_f1s = [], []
        
        missing_data = False
        
        # Katalog z wynikami dla danej głębokości
        depth_dir = f'depth_{depth}'
        
        for fold in range(1, K_FOLDS + 1):
            fold_dir = f'{depth_dir}/fold_{fold}'
            test_csv = f'{fold_dir}/test_data.csv'
            
            try:
                test_df = pd.read_csv(test_csv)
            except FileNotFoundError:
                print(f"Brak pliku {test_csv}. Pomijam ten fold.")
                missing_data = True
                break
                
            X_test_scaled = test_df.drop('target', axis=1).values
            y_test = test_df['target'].values
            
            X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)
            
            # Inicjalizacja JEDNEGO, czystego (bezszumowego) modelu dla danej głębokości
            current_ansatz = ansatz_Odra(NUM_QUBITS, depth)
            model_eval = HybridModel(current_ansatz, NUM_QUBITS)
            
            # --- 1. Ewaluacja wag Modelu Idealnego ---
            ideal_model_path = f'{fold_dir}/fold_{fold}_depth_{depth}_best_weights.pth'
            try:
                model_eval.load_state_dict(torch.load(ideal_model_path, map_location=torch.device('cpu'), weights_only=True))
                model_eval.eval()
                with torch.no_grad():
                    preds_ideal = (torch.round(model_eval(X_test_tensor), decimals=5) > 0).float() * 2 - 1
                    preds_ideal_np = preds_ideal.numpy().flatten()
                    ideal_accs.append(accuracy_score(y_test, preds_ideal_np))
                    ideal_f1s.append(f1_score(y_test, preds_ideal_np, pos_label=1))
            except FileNotFoundError:
                print(f"Brak wag idealnych: {ideal_model_path}")
                missing_data = True
                break
                
            # --- 2. Ewaluacja wag Modelu Zaszumionego (na czystym modelu bez szumu) ---
            noise_model_path = f'{fold_dir}/fold_{fold}_depth_{depth}_noise_best_weights.pth'
            try:
                # Ładujemy wagi trenowane z szumem do czystego modelu HybridModel
                model_eval.load_state_dict(torch.load(noise_model_path, map_location=torch.device('cpu'), weights_only=True))
                model_eval.eval()
                with torch.no_grad():
                    preds_noise = (torch.round(model_eval(X_test_tensor), decimals=5) > 0).float() * 2 - 1
                    preds_noise_np = preds_noise.numpy().flatten()
                    noisy_accs.append(accuracy_score(y_test, preds_noise_np))
                    noisy_f1s.append(f1_score(y_test, preds_noise_np, pos_label=1))
            except FileNotFoundError:
                print(f"Brak wag zaszumionych: {noise_model_path}")
                missing_data = True
                break

        if missing_data or not ideal_accs or not noisy_accs:
            print(f"-> Pominięto generowanie wierszy dla głębokości {depth} z powodu brakujących danych.\n")
            continue

        # --- Obliczenia Średnich i Odchyleń ---
        mean_ideal_acc, std_ideal_acc = np.mean(ideal_accs), np.std(ideal_accs)
        mean_ideal_f1, std_ideal_f1 = np.mean(ideal_f1s), np.std(ideal_f1s)
        
        mean_noisy_acc, std_noisy_acc = np.mean(noisy_accs), np.std(noisy_accs)
        mean_noisy_f1, std_noisy_f1 = np.mean(noisy_f1s), np.std(noisy_f1s)
        
        # Różnica procentowa (w punktach procentowych)
        diff_acc_pct = (mean_noisy_acc - mean_ideal_acc) * 100
        diff_f1_pct = (mean_noisy_f1 - mean_ideal_f1) * 100
        
        # --- Dodanie wierszy do wspólnej tabeli Markdown ---
        markdown_table += f"| **{depth}** | **Mean Accuracy** | **{mean_ideal_acc:.4f}** ± {std_ideal_acc:.4f} | **{mean_noisy_acc:.4f}** ± {std_noisy_acc:.4f} | **{diff_acc_pct:+.2f}%** |\n"
        markdown_table += f"| **{depth}** | **Mean F1 Score** | **{mean_ideal_f1:.4f}** ± {std_ideal_f1:.4f} | **{mean_noisy_f1:.4f}** ± {std_noisy_f1:.4f} | **{diff_f1_pct:+.2f}%** |\n"

        # --- Obliczenie sumy acc + f1 ---
        sum_ideal = mean_ideal_acc + mean_ideal_f1
        sum_noisy = mean_noisy_acc + mean_noisy_f1
        diff_sum = sum_noisy - sum_ideal
        
        # Dodanie wiersza rozdzielonego tabulatorami (\t) do podsumowania Excel
        summary_lines += f"{depth}\t{sum_ideal:.4f}\t{sum_noisy:.4f}\t{diff_sum:.4f}\n"

    # Zamknięcie bloku kodu tekstowego z podsumowaniem
    summary_lines += "```\n"

    # Wyświetlenie pełnej, połączonej tabeli oraz podsumowania
    display(Markdown(markdown_table + summary_lines))

Wykryte głębokości: [2, 4, 6]
Brak wag zaszumionych: depth_4/fold_1/fold_1_depth_4_noise_best_weights.pth
-> Pominięto generowanie wierszy dla głębokości 4 z powodu brakujących danych.

Brak wag zaszumionych: depth_6/fold_1/fold_1_depth_6_noise_best_weights.pth
-> Pominięto generowanie wierszy dla głębokości 6 z powodu brakujących danych.



| Głębokość (Depth) | Metryka | Model Idealny (Noiseless) | Model Zaszumiony (Noisy) | Różnica (Noisy vs Idealny) |
| :--- | :--- | :--- | :--- | :--- |
| **2** | **Mean Accuracy** | **0.8870** ± 0.0247 | **0.8703** ± 0.0254 | **-1.68%** |
| **2** | **Mean F1 Score** | **0.8700** ± 0.0293 | **0.8494** ± 0.0272 | **-2.06%** |

### Podsumowanie (Mean Accuracy + Mean F1 Score)
Gotowe do przekopiowania do Excela:
```text
Depth	Ideal_Sum	Noisy_Sum	Diff_Sum
2	1.7571	1.7197	-0.0374
```
